# 05 — Validation design and baseline models

**Question:** how should this dataset be split, and what does a sensible model achieve?

This is the most important notebook in the project.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
from smartnet import config
from smartnet.data import loader
from smartnet.evaluation.splits import summarise_strategies, make_splitter, split_diagnostics
from smartnet.models.experiment import run_sweep, leakage_table

df = loader.load_analysis_frame()

## The leakage risk

Each labelled motion is a contiguous run of 1-second epochs, and each row carries features for the 10 seconds either side. Adjacent rows within an event overlap in most of their input. Splitting rows at random puts near-duplicates on both sides.

In [ ]:
diag = summarise_strategies(df, 'motion_5cat')
diag[['strategy','grouping','shared_events','pct_test_from_seen_event']]

**Under a random split, 59.6% of test epochs come from a motion event that also appears in training.** Grouped splits reduce this to zero by construction.

In [ ]:
from smartnet.visualization import plots as P
fig = P.plot_leakage_diagnostic(diag); plt.show()

## Does it actually matter?

The diagnostic establishes the *risk*. Whether it inflates scores is an empirical question.

In [ ]:
results, artefacts = run_sweep(df, label_col='motion_5cat')
leakage_table(results)

### The hypothesis was wrong, and that is the finding

Inflation is **0.3 to 1.0 percentage points** — small. The structural risk is real and measurable, but on this dataset it does not materially change the score.

The likely reason: the classes are separated by broad differences in signal energy that generalise across events and days, not by event-specific quirks a model could memorise.

Reporting this honestly is worth more than the tidier story. The design is still correct — the diagnostic is cheap, the risk was genuine, and on a dataset with longer events or subtler classes the answer could easily have gone the other way. **Grouped-event CV is used for every subsequent result.**

In [ ]:
base = results[results.model=='majority_baseline']
print('Majority baseline (grouped-event):')
print(base[base.strategy=='grouped_event'][['cv_accuracy_mean','cv_balanced_accuracy_mean']].round(4).to_string(index=False))
print('\nA model predicting only the most common class reaches 37% accuracy.')
print('Every number below must be read against that floor.')

In [ ]:
fig = P.plot_model_comparison(results); plt.show()